In [ ]:
import os
import numpy as np
import pandas as pd
import gc
import torch

def extract_clare_data(base_path='./', num_subjects=5):
    eeg_dir = os.path.join(base_path, 'EEG')
    labels_dir = os.path.join(base_path, 'Labels')
    
    subject_folders = sorted([f for f in os.listdir(eeg_dir) if os.path.isdir(os.path.join(eeg_dir, f))])
    subject_folders = subject_folders[:num_subjects]
    
    X_list, y_list, meta_list = [], [], []
    fs = 256
    samples_per_epoch = 10 * fs 
    
    print(f"Extracting CLARE Workload data for {len(subject_folders)} subjects...")
    
    for subject_idx, subject_id in enumerate(subject_folders):
        print(f"Subject {subject_id}...")    
        
        subj_eeg_path = os.path.join(eeg_dir, subject_id)
        label_file = os.path.join(labels_dir, f"{subject_id}.csv")
        
        if not os.path.exists(label_file):
            print(f"  [!] Label file {subject_id}.csv not found. Skipping subject.")
            continue
            
        try:
            # Load the entire label file for the subject
            df_labels = pd.read_csv(label_file)
            labels_all = df_labels['level_3'].values
            
            # Pointer to track labels while analyzing different experiments
            label_idx = 0 
            
            # Iterate through the 4 experiments
            for exp_num in range(4):
                eeg_file = os.path.join(subj_eeg_path, f"eeg_data_exp_{exp_num}.csv")
                
                if not os.path.exists(eeg_file):
                    continue
                    
                df_eeg = pd.read_csv(eeg_file)
                # Extract only the 4 required channels
                eeg_data = df_eeg[['TP9', 'AF7', 'AF8', 'TP10']].values
                
                n_epochs = eeg_data.shape[0] // samples_per_epoch
                
                for i in range(n_epochs):
                    if label_idx >= len(labels_all):
                        break
                        
                    # extract the 10-second block and transpose
                    start_idx = i * samples_per_epoch
                    epoch_data = eeg_data[start_idx : start_idx + samples_per_epoch, :].T
                    
                    # binarization: 1-4 (Low) -> 0, 5-9 (High) -> 1
                    score = labels_all[label_idx]
                    binary_label = 0 if score < 5 else 1
                    
                    X_list.append(epoch_data.astype('float32'))
                    y_list.append(binary_label)
                    meta_list.append(subject_idx)
                    
                    label_idx += 1
                    
        except Exception as e:
            print(f"  [ERROR] in {subject_id}: {e}")
            
        gc.collect()

    if not X_list:
        print("\nNo data extracted. Check the structure and file paths.")
        return None, None, None

    X = np.stack(X_list, axis=0)
    y = np.array(y_list, dtype=np.int64)
    metadata = pd.DataFrame({'subject': meta_list})
    
    X_tensor = torch.from_numpy(X)
    y_tensor = torch.from_numpy(y)
    
    return X_tensor, y_tensor, metadata

# execution
base_path = './'
X_tensor, y_tensor, metadata = extract_clare_data(base_path=base_path, num_subjects=5)

if X_tensor is not None:
    print(f"\nLoading Completed")
    print(f"X_tensor shape: {X_tensor.shape} -> (trials, 4 channels, 2560 samples)")
    print(f"y_tensor shape: {y_tensor.shape}")

    class_counts = np.bincount(y_tensor.numpy())
    print(f"\nWorkload Binarization Analysis:")
    print(f"Low Workload (Class 0) : {class_counts[0]} trials")
    print(f"High Workload (Class 1): {class_counts[1]} trials")

Extracting CLARE Workload data for 5 subjects...
Subject 1026...
Subject 1106...
Subject 1175...
Subject 1194...
Subject 1337...

Loading Completed
X_tensor shape: torch.Size([270, 4, 2560]) -> (trials, 4 channels, 2560 samples)
y_tensor shape: torch.Size([270])

Workload Binarization Analysis:
Low Workload (Class 0) : 28 trials
High Workload (Class 1): 242 trials


In [ ]:
import torch
import torch.optim as optim
import torch.nn as nn
from sklearn.metrics import accuracy_score
import time
import numpy as np
from utils import calculate_advanced_metrics, plot_confusion_matrix
from EEGNet import EEGNet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

X_tensor_gpu = X_tensor.float().to(device)
y_tensor_gpu = y_tensor.long().to(device)

n_channels = X_tensor_gpu.shape[1]
n_samples = X_tensor_gpu.shape[2]
n_classes = 2

class_names = ['Low Workload', 'High Workload']

# calculate class weights for unbalanced dataset
class_counts = np.bincount(y_tensor.numpy())
weights = 1.0 / torch.tensor(class_counts, dtype=torch.float32)
weights = weights / weights.sum() * n_classes
weights = weights.to(device)
criterion = nn.CrossEntropyLoss(weight=weights)

print(f"Weights applied to Loss: Low Workload={weights[0]:.4f}, High Workload={weights[1]:.4f}")

# test with Latent Alignment
print("\n" + "="*60)
print("=== Start LOSO Validation CLARE (EEGNet + LATENT ALIGNMENT) ===")
print("="*60)

all_y_true_align = []
all_y_pred_align = []
loso_results_align = []
subjects_array = metadata['subject'].unique()
num_epochs = 30 

for test_subject in subjects_array:
    print(f"\n-> Training for Test Subject: {test_subject}")
    
    train_mask = (metadata['subject'] != test_subject).values
    test_mask = (metadata['subject'] == test_subject).values
    
    model = EEGNet(in_shape=(n_channels, n_samples), n_out=n_classes, alignment='latent').to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    train_subjects = metadata[train_mask]['subject'].unique()
    
    model.train()
    start_time = time.time()
    
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        for subj in train_subjects:
            subj_mask = (metadata['subject'] == subj).values
            batch_X = X_tensor_gpu[subj_mask]
            batch_y = y_tensor_gpu[subj_mask]
            
            optimizer.zero_grad(set_to_none=True)
            
            outputs = model(batch_X, sbj_trials=batch_X.shape[0])
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            
        if (epoch + 1) % 10 == 0:
            print(f"   Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss/len(train_subjects):.4f}")
            
    # Evaluation
    model.eval()
    with torch.no_grad():
        test_X = X_tensor_gpu[test_mask]
        test_y = y_tensor_gpu[test_mask]
        
        outputs = model(test_X, sbj_trials=test_X.shape[0])
        _, predicted = torch.max(outputs.data, 1)
        
        y_true_np = test_y.cpu().numpy()
        y_pred_np = predicted.cpu().numpy()
        
        all_y_true_align.extend(y_true_np)
        all_y_pred_align.extend(y_pred_np)
        
        acc = accuracy_score(y_true_np, y_pred_np)
        loso_results_align.append(acc)
        
    print(f"-> End of Test Subject {test_subject} | Accuracy: {acc*100:.2f}% | Time: {time.time()-start_time:.1f}s")

# Latent Alignment Report
all_y_true_align = np.array(all_y_true_align)
all_y_pred_align = np.array(all_y_pred_align)

print("\n--- MODEL ANALYSIS: LATENT ALIGNMENT ---")
calculate_advanced_metrics(all_y_true_align, all_y_pred_align, class_names=class_names)
plot_confusion_matrix(all_y_true_align, all_y_pred_align, class_names=class_names, title="Cognitive Workload: EEGNet + Latent Alignment")

Using device: cpu
Pesi applicati alla Loss: Low Workload=1.7926, High Workload=0.2074

=== Start LOSO Validation CLARE (EEGNet + LATENT ALIGNMENT) ===

-> Training for Test Subject: 0


KeyboardInterrupt: 

In [ ]:
# test without Latent Alignment
print("\n" + "="*60)
print("=== Start LOSO Validation CLARE (EEGNet BASELINE) ===")
print("="*60)

all_y_true_base = []
all_y_pred_base = []
loso_results_base = []

for test_subject in subjects_array:
    print(f"\n-> Training for Test Subject: {test_subject}")
    
    train_mask = (metadata['subject'] != test_subject).values
    test_mask = (metadata['subject'] == test_subject).values
    
    X_train_loso = X_tensor_gpu[train_mask]
    y_train_loso = y_tensor_gpu[train_mask]
    
    # initialize model without alignment
    model = EEGNet(in_shape=(n_channels, n_samples), n_out=n_classes, alignment='None').to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    model.train()
    start_time = time.time()
    
    for epoch in range(num_epochs):
        optimizer.zero_grad(set_to_none=True)
        
        outputs = model(X_train_loso, sbj_trials=X_train_loso.shape[0])
        loss = criterion(outputs, y_train_loso)
        loss.backward()
        optimizer.step()
        
        if (epoch + 1) % 10 == 0:
            print(f"   Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")
            
    # evaluation
    model.eval()
    with torch.no_grad():
        test_X = X_tensor_gpu[test_mask]
        test_y = y_tensor_gpu[test_mask]
        
        outputs = model(test_X, sbj_trials=test_X.shape[0])
        _, predicted = torch.max(outputs.data, 1)
        
        y_true_np = test_y.cpu().numpy()
        y_pred_np = predicted.cpu().numpy()
        
        all_y_true_base.extend(y_true_np)
        all_y_pred_base.extend(y_pred_np)
        
        acc = accuracy_score(y_true_np, y_pred_np)
        loso_results_base.append(acc)
        
    print(f"-> End of Test Subject {test_subject} | Accuracy: {acc*100:.2f}% | Time: {time.time()-start_time:.1f}s")

# Baseline Report
all_y_true_base = np.array(all_y_true_base)
all_y_pred_base = np.array(all_y_pred_base)

print("\n--- MODEL ANALYSIS: BASELINE ---")
calculate_advanced_metrics(all_y_true_base, all_y_pred_base, class_names=class_names)
plot_confusion_matrix(all_y_true_base, all_y_pred_base, class_names=class_names, title="Cognitive Workload: EEGNet BASELINE")